# Tutorial 9 — Speech and multilingual ASR

[Lecture notes](../lectures/09_speech.md) · [Slides](../slides/09_speech.pptx) · [Exercise sheet](../exercises/09_speech.md)

**40 minutes, CPU, NumPy only.** You will turn a waveform into time-frequency frames, read a CTC output, score transcripts with WER, and measure what happens when a recogniser trained on one voice meets new voices. The audio is synthetic tones and the transcripts are written by hand. Nothing here transcribes speech or measures a real model.

| Minutes | Step |
|---|---|
| 0–5 | A waveform and its short-time Fourier transform |
| 5–10 | The sample rate is part of the data |
| 10–15 | CTC: frame probabilities and the best path |
| 15–20 | WER with an alignment you can read |
| 20–30 | One voice in training, new voices at test; cascade arithmetic |
| 30–40 | Your turn: two functions with tests |

![Illustration of a local window over a waveform.](../data/audio-window.svg)

In [1]:
import sys
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)

Python 3.13.1 NumPy 2.1.3 seed 42


## 1. A waveform and its STFT

The signal is one second of a 440 Hz tone with a little noise, sampled at 16 kHz. Frames are 400 samples long (25 ms) and start every 160 samples (10 ms). The output is a log-magnitude STFT. A mel spectrogram would add one more step, a bank of filters that pools the frequency bins.

Before running: how many frames does one second give, and how many frequency bins does a 400-sample frame give?

In [2]:
sr=16000; t=np.arange(sr)/sr
wave=.5*np.sin(2*np.pi*440*t)+.01*rng.normal(size=sr)
frame_length=400; hop=160
frames=np.stack([wave[i:i+frame_length] for i in range(0,len(wave)-frame_length+1,hop)])
spectrum=np.abs(np.fft.rfft(frames*np.hanning(frame_length),axis=1))
log_spectrum=np.log(spectrum+1e-8)
freqs=np.fft.rfftfreq(frame_length,1/sr)
peak=freqs[spectrum.mean(axis=0).argmax()]
print("seconds",len(wave)/sr,"STFT shape",spectrum.shape,"dominant Hz",peak)
assert abs(peak-440)<=sr/frame_length

seconds 1.0 STFT shape (98, 201) dominant Hz 440.0


One second becomes 98 frames of 201 bins. The bins are 16,000/400 = 40 Hz apart, so 440 Hz falls on bin 11. A three-second spoken question would be about 300 frames for perhaps eight words. That mismatch in length is the problem CTC solves in section 3.

## 2. The sample rate is part of the data

Record the same tone at 48 kHz, then hand the samples to code that assumes 16 kHz.

Before running: what duration and what pitch will the code believe it has?

In [3]:
sr48=48000; wave48=.5*np.sin(2*np.pi*440*np.arange(sr48)/sr48)
def dominant_hz(x,rate): return float(np.fft.rfftfreq(len(x),1/rate)[np.abs(np.fft.rfft(x)).argmax()])
print("read as 48 kHz: seconds",len(wave48)/48000,"dominant Hz",round(dominant_hz(wave48,48000),1))
print("relabelled as 16 kHz: seconds",len(wave48)/16000,"dominant Hz",round(dominant_hz(wave48,16000),1))
every_third=wave48[::3]      # crude resampling; safe here only because the tone is far below the new 8 kHz limit
print("resampled to 16 kHz: seconds",len(every_third)/16000,"dominant Hz",round(dominant_hz(every_third,16000),1))
assert abs(dominant_hz(wave48,16000)-440/3)<1 and abs(dominant_hz(every_third,16000)-440)<1

read as 48 kHz: seconds 1.0 dominant Hz 440.0
relabelled as 16 kHz: seconds 3.0 dominant Hz 146.7
resampled to 16 kHz: seconds 1.0 dominant Hz 440.0


Relabelling stretches the clip to three seconds and drops the pitch to 146.7 Hz, a third of the true value. Resampling keeps one second and 440 Hz. Taking every third sample works for this tone; for speech, a resampler must first remove content above 8 kHz, so use a library function.

## 3. CTC: frame probabilities and the best path

A CTC model outputs one probability distribution per frame over the characters plus a blank, written `_`. Here are four frames over `_`, `a` and `b`.

Before running: pick the most probable symbol in each frame. Collapse the path by hand: merge repeated symbols, then delete blanks.

In [4]:
symbols=["_","a","b"]
P=np.array([[.1,.8,.1],[.6,.3,.1],[.2,.1,.7],[.5,.1,.4]])     # rows are frames, columns are symbols
best=[symbols[k] for k in P.argmax(axis=1)]
print("best symbol per frame:",best,"with probability",round(float(P.max(axis=1).prod()),3))
print("number of possible paths:",len(symbols)**len(P))
assert np.allclose(P.sum(axis=1),1)

best symbol per frame: ['a', '_', 'b', '_'] with probability 0.168
number of possible paths: 81


The best path is `a _ b _` and it collapses to "ab". Its probability is 0.8 × 0.6 × 0.7 × 0.5 = 0.168. Other paths, such as `a a b b`, give "ab" too. CTC adds them all. You will compute that sum in part A of your turn, once you have written the collapse rule.

## 4. WER with an alignment you can read

Word error rate is the number of substitutions, deletions and insertions divided by the number of reference words. The edit distance finds the smallest total. The second function walks back through the table and names each operation.

Before running: align "the lab opens today" with "lab opens on monday" by hand and count the three kinds of error.

In [5]:
def edit_distance(a,b):
    D=np.zeros((len(a)+1,len(b)+1),dtype=int)
    D[:,0]=np.arange(len(a)+1); D[0,:]=np.arange(len(b)+1)
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1):
            D[i,j]=min(D[i-1,j]+1,D[i,j-1]+1,D[i-1,j-1]+(a[i-1]!=b[j-1]))
    return int(D[-1,-1])
def wer(ref,hyp):
    a,b=ref.split(),hyp.split()
    if not a:
        if not b: return 0.
        raise ValueError("WER undefined for empty reference and nonempty hypothesis")
    return edit_distance(a,b)/len(a)
ref="the lab opens today"; hyp="lab opens on monday"
print("WER",wer(ref,hyp))
assert wer(ref,hyp)==.75
assert wer("yes","yes yes yes")==2.
assert edit_distance("kitten","sitting")==3

WER 0.75


In [6]:
def alignment(ref,hyp):
    a,b=ref.split(),hyp.split()
    D=np.zeros((len(a)+1,len(b)+1),dtype=int); D[:,0]=np.arange(len(a)+1); D[0,:]=np.arange(len(b)+1)
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1):
            D[i,j]=min(D[i-1,j]+1,D[i,j-1]+1,D[i-1,j-1]+(a[i-1]!=b[j-1]))
    i,j,ops=len(a),len(b),[]
    while i>0 or j>0:
        if i>0 and j>0 and D[i,j]==D[i-1,j-1]+(a[i-1]!=b[j-1]):
            ops.append(("ok" if a[i-1]==b[j-1] else "S",a[i-1],b[j-1])); i-=1; j-=1
        elif i>0 and D[i,j]==D[i-1,j]+1: ops.append(("D",a[i-1],"")); i-=1
        else: ops.append(("I","",b[j-1])); j-=1
    return ops[::-1]
for r,h in [(ref,hyp),("habari ya asubuhi","habari za asubuhi"),("ninakupenda","nakupenda")]:
    ops=alignment(r,h)
    print(" ".join(f"{o}({x or '-'}>{y or '-'})" for o,x,y in ops))
    print("   S,D,I =",[sum(o==k for o,_,_ in ops) for k in "SDI"],"  WER =",round(wer(r,h),3),"  characters differing =",edit_distance(r,h),"of",len(r))
assert sum(o!="ok" for o,_,_ in alignment(ref,hyp))==3

D(the>-) ok(lab>lab) ok(opens>opens) I(->on) S(today>monday)
   S,D,I = [1, 1, 1]   WER = 0.75   characters differing = 9 of 19
ok(habari>habari) S(ya>za) ok(asubuhi>asubuhi)
   S,D,I = [1, 0, 0]   WER = 0.333   characters differing = 1 of 17
S(ninakupenda>nakupenda)
   S,D,I = [1, 0, 0]   WER = 1.0   characters differing = 2 of 11


The first pair has one deletion, one insertion and one substitution: 3 errors over 4 reference words. The two Swahili pairs show what WER hides. "ya" for "za" is one wrong word in three. "nakupenda" for "ninakupenda" (I love you) is two missing characters in one long word, and WER calls the whole utterance wrong. Part B of your turn adds a character error rate so that you can report both.

## 5. One voice in training, new voices at test

Task 6 subtask 1 gives synthetic spoken questions for training and development, and human recordings at test. This toy shows the shape of that problem. Each of three "words" is a pair of tones whose frequency ratio identifies the word. A "voice" scales both tones. The recogniser stores the average spectrum of each word and picks the nearest.

Before running: the recogniser is trained on one voice. Predict its accuracy on that voice and on new voices.

In [7]:
ratios=[1.5,2.0,2.5]; clip_t=np.arange(4000)/sr                       # 0.25 s clips
def magnitude(word,voice):
    f0=200*voice
    x=np.sin(2*np.pi*f0*clip_t)+np.sin(2*np.pi*f0*ratios[word]*clip_t)+.3*rng.normal(size=len(clip_t))
    return np.abs(np.fft.rfft(x*np.hanning(len(x))))
def clip(word,voice):
    s=np.log(magnitude(word,voice)+1e-3); return s-s.mean()
def ratio_guess(word,voice):                 # a feature that ignores the voice: the ratio of the two strongest peaks
    m=magnitude(word,voice); i=int(m.argmax()); m[max(i-5,0):i+6]=0; j=int(m.argmax())
    return int(np.abs(np.array(ratios)-max(i,j)/min(i,j)).argmin())
def ratio_accuracy(voices,n=300):
    w=rng.integers(0,3,n); v=rng.choice(voices,n); return float(np.mean([ratio_guess(a,b)==a for a,b in zip(w,v)]))
def dataset(voices,n):
    w=rng.integers(0,3,n); v=rng.choice(voices,n); return np.array([clip(a,b) for a,b in zip(w,v)]),w
def train(voices,n=150):
    F,w=dataset(voices,n); return np.array([F[w==k].mean(0) for k in range(3)])
def accuracy(centroids,voices,n=300):
    F,w=dataset(voices,n); return float(((F@centroids.T/np.linalg.norm(centroids,axis=1)).argmax(1)==w).mean())
new_voices=np.linspace(.8,1.3,11)
one_voice=train([1.0]); many_voices=train(np.linspace(.8,1.3,6))
print("trained on one voice    -> same voice",accuracy(one_voice,[1.0])," new voices",accuracy(one_voice,new_voices))
print("trained on six voices   -> same voice",accuracy(many_voices,[1.0])," new voices",accuracy(many_voices,new_voices))
print("peak-ratio feature      -> same voice",ratio_accuracy([1.0])," new voices",ratio_accuracy(new_voices))
shift_same,shift_new=accuracy(one_voice,[1.0]),accuracy(one_voice,new_voices)
assert shift_same>.95 and shift_new<shift_same and ratio_accuracy(new_voices)>.95

trained on one voice    -> same voice 1.0  new voices 0.36333333333333334
trained on six voices   -> same voice 0.6866666666666666  new voices 0.5
peak-ratio feature      -> same voice 1.0  new voices 1.0


Line 1: the recogniser is perfect on its training voice and near chance (1 in 3) on new voices. A development score on audio like the training audio tells you about the first number only.

Line 2: six training voices raise new-voice accuracy and lower accuracy on the original voice. One average spectrum per word cannot describe six voices, so more data did not fix a model that is too simple.

Line 3: a feature that is the same for every voice needs no voice-matched training data. Self-supervised speech encoders trained on many speakers are the learned version of this idea, which is the argument for starting from one instead of training on synthetic audio alone.

For Task 6, record a small set of development questions with consenting speakers and score on it. The same result is the reason to split speech data by speaker: clips from one speaker on both sides of a split measure the first column.

### Cascade arithmetic

A cascade transcribes the question, then answers it from the text and the image. Suppose the answer stage is right 60% of the time when the transcript is right and 20% when it is wrong. These two rates are assumptions for the exercise.

In [8]:
def cascade(p_asr,p_right=.6,p_wrong=.2): return p_asr*p_right+(1-p_asr)*p_wrong
for p_asr in [1.0,.9,.7]:
    sim_asr=rng.random(20000)<p_asr
    sim=np.where(sim_asr,rng.random(20000)<.6,rng.random(20000)<.2).mean()
    print(f"questions transcribed correctly {p_asr:.1f} -> answers correct {cascade(p_asr):.2f} (simulated {sim:.3f})")
print("8-word question, 5% word error rate, independent errors: whole question correct",round(.95**8,3))
assert np.isclose(cascade(.9),.56)

questions transcribed correctly 1.0 -> answers correct 0.60 (simulated 0.593)
questions transcribed correctly 0.9 -> answers correct 0.56 (simulated 0.564)
questions transcribed correctly 0.7 -> answers correct 0.48 (simulated 0.486)
8-word question, 5% word error rate, independent errors: whole question correct 0.663


With transcripts right 90% of the time the cascade answers 0.9 × 0.6 + 0.1 × 0.2 = 0.56 correctly. At 70% it answers 0.48. The last line shows why "question transcribed correctly" is a hard target: a 5% word error rate leaves only two in three eight-word questions fully correct, if errors are independent.

## 6. Your turn

Replace each `raise` with your code, then run the test cell under it. Do not edit the tests.

**A. The CTC collapse rule.** Merge consecutive repeated symbols, then remove blanks. A blank between two equal symbols keeps both: the Hausa greeting "sannu" needs one between its two n's.

In [ ]:
# YOUR TURN
def ctc_collapse(path, blank="_"):
    """Collapse a sequence of symbols by the CTC rule and return the result as a string."""
    raise NotImplementedError("your turn")

In [10]:
assert ctc_collapse(["a","a"])=="a"
assert ctc_collapse(["a","_","a"])=="aa"
assert ctc_collapse(["a","a","_","a","b","b","_"])=="aab"
assert ctc_collapse("ssa_nn_nuu")=="sannu" and ctc_collapse("ssannnuu")=="sanu"
assert ctc_collapse("____")=="" and ctc_collapse("0aa0b",blank="0")=="ab"
assert ctc_collapse(best)=="ab"
# Sum the probability of every one of the 81 paths from section 3 that collapses to "ab".
from itertools import product
paths=[p for p in product(range(3),repeat=len(P)) if ctc_collapse([symbols[k] for k in p])=="ab"]
total=sum(np.prod([P[t,k] for t,k in enumerate(p)]) for p in paths)
print("paths that collapse to 'ab':",len(paths)," total probability",round(float(total),4)," best path alone 0.168")
print("A passed")

paths that collapse to 'ab': 15  total probability 0.6041  best path alone 0.168
A passed


CTC training maximises the total over all those paths, with a dynamic programme instead of a list. Greedy decoding reports only the best path. How much of the probability of "ab" did the best path miss?

**B. Character error rate.** Edit distance over characters, divided by the number of reference characters. Count spaces as characters. Follow the rule `wer` uses for an empty reference: return 0.0 if the hypothesis is empty too, and raise `ValueError` otherwise.

In [ ]:
# YOUR TURN
def cer(ref, hyp):
    """Character error rate with the empty-reference rule described above."""
    raise NotImplementedError("your turn")

In [12]:
assert cer("kitten","sitting")==3/6
assert cer("ninakupenda","nakupenda")==2/11 and wer("ninakupenda","nakupenda")==1.0
assert cer("","")==0.0 and cer("abc","abc")==0.0 and cer("ab","abab")==1.0
try:
    cer("","a"); raise AssertionError("an empty reference with a non-empty hypothesis must raise ValueError")
except ValueError:
    pass
for r,h in [("habari ya asubuhi","habari za asubuhi"),("Habari ya asubuhi","habari ya asubuhi"),("ƙofa","kofa")]:
    print(f"{r!r} vs {h!r}: WER {wer(r,h):.3f}  CER {cer(r,h):.3f}")
print("B passed")

'habari ya asubuhi' vs 'habari za asubuhi': WER 0.333  CER 0.059
'Habari ya asubuhi' vs 'habari ya asubuhi': WER 0.333  CER 0.059
'ƙofa' vs 'kofa': WER 1.000  CER 0.250
B passed


The second pair differs only in the capital H. Lower-casing both sides before scoring removes that error, and most people would accept the rule. The third pair is the Hausa word "ƙofa" (door) written with a plain k. A rule that maps ƙ to k would remove that error too, and it would hide a consonant the language distinguishes. Write your normalisation rules down before you score, and report raw and normalised numbers.

## Discussion

1. Why can WER exceed 100%?
2. What goes wrong when a 48 kHz recording is treated as 16 kHz without resampling?
3. Why should two clips from one speaker stay in the same split?

## From the toy to Task 6

Subtask 1 gives an image and a spoken question. A cascade needs an ASR model for your language and the text system from subtask 2. An end-to-end model takes the audio directly. Either way the test audio is human and your training audio is synthetic, so record a few dozen development questions with consenting speakers and score both designs on them. `course/extensions/hf_audio.py --audio path.wav` runs a pretrained Whisper on a clip you are allowed to use; see the [extension guide](../extensions/README.md).

Submit this notebook with both tests passing and a 150–250-word note: the two accuracies from section 5, what they imply for your evaluation plan, and one way the tone toy differs from speech.